# CNN t1 quantizada (8 bits) no SAT-6 

Notebook enxuto contendo apenas o necessário para:
1. Carregar o dataset SAT-6 (já "pickled" pelo `0_pickle_dataset.ipynb`);
2. Definir e treinar a topologia **t1** com quantização de **8 bits** (pesos e ativações);
3. Testar/avaliar a acurácia do modelo;
4. Exportar o modelo para **QONNX**, formato de entrada aceito pelo **hls4ml** (ao contrário do FINN, não é necessário aplicar as transformações de streamlining/conversão para hardware — isso é feito pelo próprio hls4ml a partir do ONNX/QONNX).

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from tqdm import tqdm

from dependencies import value
from brevitas.core.bit_width import BitWidthImplType
from brevitas.core.quant import QuantType
from brevitas.core.restrict_val import FloatToIntImplType, RestrictValueType
from brevitas.core.scaling import ScalingImplType
from brevitas.core.zero_point import ZeroZeroPoint
from brevitas.inject import ExtendedInjector
from brevitas.quant.solver import ActQuantSolver, WeightQuantSolver
from brevitas.quant import Uint8ActPerTensorFloat
import brevitas.nn as qnn
from brevitas.export import export_qonnx

from utils.utils import read_object
from cnn_base import t1_quantizedCNN

%matplotlib inline
torch.manual_seed(4)

## 1. Dataset

A classe `SatImgDataset` precisa estar definida antes de desserializar os `.pkl` (mesmo nome/estrutura usada em `0_pickle_dataset.ipynb`).

In [2]:
batch_size = 512
epochs = 30
lr = 3e-4
weight_bits = 8  
device = torch.device('cpu')

train_dataset_output_name = "./dataset/train_dataset.pkl"
test_dataset_output_name = "./dataset/test_dataset.pkl"

dataset_train = read_object(train_dataset_output_name)
loader_train = DataLoader(dataset_train, batch_size, shuffle=True)

dataset_test = read_object(test_dataset_output_name)
loader_test = DataLoader(dataset_test, batch_size, shuffle=False)

## 2. Treinamento

In [3]:
def train_model(model, loader_train, optimizer, criterion, epochs, device):
    for e in range(epochs):
        model.train()
        running_loss = 0.0
        print(f"epoch ({e})")
        for x,y in tqdm(loader_train):
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.long)

            pred = model(x)
            loss = criterion(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        mean_loss = running_loss / len(loader_train)
        print(f"epoch ({e}) loss: {mean_loss:.4f}")

    return model


model = t1_quantizedCNN(bit_quantization=weight_bits).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

model_output_filename = f"./pytorch_models/sat6-cnn-t1w{weight_bits}.pt"
model = train_model(model, loader_train, optimizer, criterion, epochs, device)

os.makedirs("./pytorch_models", exist_ok=True)
torch.save(model.state_dict(), model_output_filename)
print(f"modelo salvo em {model_output_filename}")

epoch (0)


  0%|          | 0/633 [00:00<?, ?it/s]/home/fauri/Documentos/hls4ml/.venv_hls4ml_p/lib/python3.10/site-packages/torch/_tensor.py:1394: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1908.)
  return super().rename(names)
100%|██████████| 633/633 [00:16<00:00, 38.44it/s]


epoch (0) loss: 0.4691
epoch (1)


100%|██████████| 633/633 [00:11<00:00, 57.19it/s]


epoch (1) loss: 0.1798
epoch (2)


100%|██████████| 633/633 [00:11<00:00, 54.65it/s]


... (51 outputs left out)

100%|██████████| 633/633 [00:10<00:00, 59.08it/s]


epoch (28) loss: 0.0750
epoch (29)


100%|██████████| 633/633 [00:10<00:00, 60.14it/s]

epoch (29) loss: 0.0761
modelo salvo em ./pytorch_models/sat6-cnn-t1w8.pt


## 4. Teste / avaliação

In [4]:
def test_model(model, loader_test, dataset_test, device):
    model.eval()
    with torch.no_grad():
        correct = 0
        for x,y in tqdm(loader_test):
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.long)

            pred = model(x)
            predicted = torch.max(pred, 1)[1]
            correct += (predicted == y).sum()
    accuracy = correct.item() / len(dataset_test) * 100
    print(f"acurácia do modelo t1w{weight_bits}: {accuracy}% ({len(dataset_test)} casos de teste)")
    return accuracy


test_model(model, loader_test, dataset_test, device)

100%|██████████| 159/159 [00:00<00:00, 165.61it/s]

acurácia do modelo t1w8: 96.75432098765432% (81000 casos de teste)


96.75432098765432

## 5. Exportação para QONNX (entrada do hls4ml)

Diferente do fluxo com o FINN (que exige as transformações de *streamlining*, *lowering* e conversão para camadas de hardware antes da síntese), o **hls4ml** consegue consumir o modelo diretamente a partir do QONNX exportado pelo Brevitas, aplicando internamente o parsing das camadas quantizadas (`Quant`, `QuantConv`, `QuantLinear`, etc.). Portanto, basta exportar e limpar o grafo ONNX.

In [7]:
import qonnx.util.cleanup
import onnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.gemm_to_matmul import GemmToMatMul
from qonnx.util.to_channels_last import to_channels_last


os.makedirs("./hls4ml_onnxs", exist_ok=True)
export_onnx_path = f"./hls4ml_onnxs/sat6-cnn-t1w{weight_bits}_teste.onnx"

model.eval()
export_qonnx(model,  torch.randn(1, 4, 32, 32), export_path=export_onnx_path, opset_version=14)
print(f"modelo QONNX exportado em {export_onnx_path}")

# Clean and convert Gemm → MatMul
qonnx.util.cleanup.cleanup(export_onnx_path, out_file=export_onnx_path)
#model_wrapper = ModelWrapper(export_onnx_path)
#model_wrapper = model_wrapper.transform(GemmToMatMul())
#model_wrapper = qonnx.util.cleanup.cleanup_model(model_wrapper)
#model_wrapper.save(export_onnx_path)
#print(f'Cleaned QONNX saved to {export_onnx_path}')

modelo QONNX exportado em ./hls4ml_onnxs/sat6-cnn-t1w8_teste.onnx


In [8]:
to_channels_last(
    in_file=export_onnx_path,
    out_file=export_onnx_path,
    make_input_channels_last=True
)
print(f'Model layout converted to channels_last and saved to {export_onnx_path}')

Model layout converted to channels_last and saved to ./hls4ml_onnxs/sat6-cnn-t1w8_teste.onnx


In [10]:
onnx_model = onnx.load(export_onnx_path)

# Caso precisar, mas melhor não usar.
def fix_onnx_node_names(model_path, output_path):
    model = onnx.load(model_path)
    graph = model.graph
    
    # 1. Corrige os nós do grafo
    for i, node in enumerate(graph.node):
        if not node.name:  # Se o nome estiver vazio ("")
            # Cria um nome baseado no tipo do operador e no índice
            node.name = f"{node.op_type.lower()}_{i}"
            
    # 2. Garante que os tensores intermediários (valores) também tenham nome se necessário
    for i, value_info in enumerate(graph.value_info):
        if not value_info.name:
            value_info.name = f"tensor_{i}"
            
    onnx.save(model, output_path)
    print(f"Modelo corrigido salvo em: {output_path}")


fix_onnx_node_names(export_onnx_path, export_onnx_path)

Modelo corrigido salvo em: ./hls4ml_onnxs/sat6-cnn-t1w8_teste.onnx
